In [6]:
from pathlib import Path
from rdflib import Graph

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"

print("Data directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.exists())

files = [
    "wittgensteiniana_final.ttl",
    "wittgensteiniana_comments.ttl",
    "wab_tractatus_subgraph.ttl",
    "wab_tractatus_comments.ttl"
]

for filename in files:
    path = DATA_DIR / filename
    print(filename, ":", path.exists())

witt = Graph()

for filename in files[:2]:
    path = DATA_DIR / filename
    witt.parse(
        data=path.read_text(encoding="utf-8"),
        format="turtle",
        publicID=path.as_uri()
    )

wab = Graph()

for filename in files[2:]:
    path = DATA_DIR / filename
    wab.parse(
        data=path.read_text(encoding="utf-8"),
        format="turtle",
        publicID=path.as_uri()
    )

print("\nWittgensteiniana triples:", len(witt))
print("WAB triples:", len(wab))

Data directory: c:\Users\Lenovo\Documents\GitHub\owl-align\tractatus-alignment\data
Directory exists: True
wittgensteiniana_final.ttl : True
wittgensteiniana_comments.ttl : True
wab_tractatus_subgraph.ttl : True
wab_tractatus_comments.ttl : True

Wittgensteiniana triples: 15588
WAB triples: 27270


In [ ]:
from rdflib import RDF, RDFS, OWL, SKOS, URIRef
from urllib.parse import unquote

def local_name(uri):
    return unquote(str(uri).split("#")[-1].split("/")[-1])

def get_values(graph, subject, predicate):
    return sorted({
        str(value).strip()
        for value in graph.objects(subject, predicate)
        if str(value).strip()
    })

def build_entity_profiles(graph):
    entity_types = {
        OWL.Class: "Class",
        RDFS.Class: "Class",
        OWL.ObjectProperty: "Object property",
        OWL.DatatypeProperty: "Datatype property"
    }

    entities = set()

    for entity_type in entity_types:
        entities.update(graph.subjects(RDF.type, entity_type))

    profiles = {}

    for entity in sorted(entities, key=str):
        if not isinstance(entity, URIRef):
            continue

        types = sorted({
            label
            for entity_type, label in entity_types.items()
            if (entity, RDF.type, entity_type) in graph
        })

        labels = get_values(graph, entity, RDFS.label)
        comments = [
            comment
            for comment in get_values(graph, entity, RDFS.comment)
            if not comment.startswith(
                "Derived from the WAB ontology axioms"
            )
        ]
        synonyms = get_values(graph, entity, SKOS.altLabel)

        parts = [
            f"Entity: {'; '.join(labels) if labels else local_name(entity)}",
            f"Type: {', '.join(types)}"
        ]

        parts.extend(
            f"Description: {comment}"
            for comment in comments
        )

        parts.extend(
            f"Alternative name: {synonym}"
            for synonym in synonyms
        )

        profiles[str(entity)] = ". ".join(parts)

    return profiles

In [15]:
witt_profiles = build_entity_profiles(witt)
wab_profiles = build_entity_profiles(wab)

print("Wittgensteiniana entities:", len(witt_profiles))
print("WAB entities:", len(wab_profiles))

Wittgensteiniana entities: 21
WAB entities: 21


In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
import numpy as np

witt_uris = list(witt_profiles.keys())
wab_uris = list(wab_profiles.keys())

witt_texts = [witt_profiles[uri] for uri in witt_uris]
wab_texts = [wab_profiles[uri] for uri in wab_uris]

witt_embeddings = model.encode(
    witt_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

wab_embeddings = model.encode(
    wab_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
similarity_matrix = np.dot(witt_embeddings, wab_embeddings.T)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (21, 21)


In [18]:
import pandas as pd

results = []

for i, witt_uri in enumerate(witt_uris):
    for j, wab_uri in enumerate(wab_uris):
        results.append({
            "Wittgensteiniana": local_name(witt_uri),
            "WAB": local_name(wab_uri),
            "Similarity": similarity_matrix[i, j]
        })

similarity_df = pd.DataFrame(results)

similarity_df = similarity_df.sort_values(
    by="Similarity",
    ascending=False
).reset_index(drop=True)

similarity_df.head(20)

,Wittgensteiniana,WAB,Similarity
0,isPartOf,isPartOf,0.766550
1,mentions,refersTo,0.763941
2,translationOfWork,language,0.761827
3,name,refersTo,0.745447
4,hasAuthor,hasAuthor,0.740970
5,name,madeBefore,0.726035
6,inDefinedTermSet,refersTo,0.716083
7,name,madeAfter,0.712693
8,inLanguage,language,0.708180
9,translationOfWork,refersTo,0.701083


In [19]:
def get_entity_type(profile):
    return profile.split(". Type: ", 1)[1].split(".", 1)[0]


type_groups = [
    "Class",
    "Object property",
    "Datatype property"
]

grouped_results = {}

for entity_type in type_groups:
    witt_indices = [
        i for i, uri in enumerate(witt_uris)
        if get_entity_type(witt_profiles[uri]) == entity_type
    ]

    wab_indices = [
        j for j, uri in enumerate(wab_uris)
        if get_entity_type(wab_profiles[uri]) == entity_type
    ]

    rows = []

    for i in witt_indices:
        for j in wab_indices:
            rows.append({
                "Wittgensteiniana": local_name(witt_uris[i]),
                "WAB": local_name(wab_uris[j]),
                "Similarity": similarity_matrix[i, j]
            })

    df = pd.DataFrame(rows).sort_values(
        "Similarity",
        ascending=False
    ).reset_index(drop=True)

    grouped_results[entity_type] = df

    print(f"\n{'=' * 60}")
    print(entity_type)
    display(df.head(15))


Class


,Wittgensteiniana,WAB,Similarity
0,Chapter,TS,0.667729
1,DefinedTermSet,TS,0.654947
2,Thesis,TS,0.651840
3,Book,TS,0.639501
4,Author,Source,0.638788
5,CreativeWork,Werk,0.630081
6,Chapter,Source,0.623993
7,CreativeWork,Source,0.605663
8,CreativeWork,TS,0.600874
9,Book,WittgensteinSource,0.597061



Object property


,Wittgensteiniana,WAB,Similarity
0,isPartOf,isPartOf,0.766550
1,mentions,refersTo,0.763941
2,translationOfWork,language,0.761827
3,hasAuthor,hasAuthor,0.740970
4,inDefinedTermSet,refersTo,0.716083
5,translationOfWork,refersTo,0.701083
6,translationOfWork,isWorkPublishedFrom,0.660545
7,inDefinedTermSet,isPartOf,0.656983
8,translationOfWork,isPartOf,0.653272
9,translationOfWork,isPublishedInWork,0.647550



Datatype property


,Wittgensteiniana,WAB,Similarity
0,name,madeBefore,0.726035
1,name,madeAfter,0.712693
2,text,madeAfter,0.648083
3,text,madeBefore,0.643475
4,description,madeBefore,0.599228
5,description,madeAfter,0.595248
6,name,hasFacsView,0.592140
7,text,hasFacsView,0.579399
8,inLanguage,madeBefore,0.578075
9,position,madeAfter,0.575149


In [20]:
mutual_matches = {}

for entity_type, df in grouped_results.items():

    best_witt = (
        df.sort_values("Similarity", ascending=False)
        .drop_duplicates(subset="Wittgensteiniana")
    )

    best_wab = (
        df.sort_values("Similarity", ascending=False)
        .drop_duplicates(subset="WAB")
    )

    mutual = best_witt.merge(
        best_wab[["Wittgensteiniana", "WAB"]],
        on=["Wittgensteiniana", "WAB"],
        how="inner"
    )

    mutual = mutual.sort_values(
        "Similarity",
        ascending=False
    ).reset_index(drop=True)

    mutual_matches[entity_type] = mutual

    print(f"\n{entity_type}: mutual best matches")
    display(mutual)


Class: mutual best matches


,Wittgensteiniana,WAB,Similarity
0,Chapter,TS,0.667729
1,Author,Source,0.638788
2,CreativeWork,Werk,0.630081



Object property: mutual best matches


,Wittgensteiniana,WAB,Similarity
0,isPartOf,isPartOf,0.766550
1,mentions,refersTo,0.763941
2,translationOfWork,language,0.761827
3,hasAuthor,hasAuthor,0.740970



Datatype property: mutual best matches


,Wittgensteiniana,WAB,Similarity
0,name,madeBefore,0.726035


In [21]:
import json

reference_path = DATA_DIR / "reference_alignment.json"

with open(reference_path, "r", encoding="utf-8") as file:
    reference_data = json.load(file)

print(json.dumps(reference_data, indent=4, ensure_ascii=False))

{
    "meta": {
        "title": "Reference alignment, WAB -> Wittgensteiniana (Pasin), draft v1",
        "created": "2026-11-03",
        "status": "draft: manual, semi-formal, revised on review",
        "source_ontology": "tractatus-alignment/data/wab_tractatus_subgraph.ttl",
        "target_ontology": "tractatus-alignment/data/wittgensteiniana_final.ttl",
        "grounding": "class/property definitions and rdfs:comment content documented in the comment overlays (wab_tractatus_comments.ttl, wittgensteiniana_comments.ttl), plus instance-level samples from both datasets",
        "grading_note": "class-level and property-level scores are graded separately; 'instances' lists grounding evidence"
    },
    "class_mappings": [
        {
            "id": "C1",
            "source": "http://purl.org/wittgensteinsource/ont/Werk",
            "target": "https://schema.org/Book",
            "strength": "strong",
            "justification": "WAB: a publication of a Wittgenstein primary so

In [22]:
def calculate_metrics(predicted, reference):
    true_positives = predicted & reference
    false_positives = predicted - reference
    false_negatives = reference - predicted

    precision = (
        len(true_positives) / len(predicted)
        if predicted else 0
    )

    recall = (
        len(true_positives) / len(reference)
        if reference else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    return {
        "TP": len(true_positives),
        "FP": len(false_positives),
        "FN": len(false_negatives),
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Correct pairs": sorted(true_positives),
        "False positives": sorted(false_positives),
        "Missed pairs": sorted(false_negatives)
    }


reference_classes = {
    (local_name(item["target"]), local_name(item["source"]))
    for item in reference_data["class_mappings"]
}

reference_properties = {
    (local_name(item["target"]), local_name(item["source"]))
    for item in reference_data["property_mappings"]
}

predicted_classes = {
    (row["Wittgensteiniana"], row["WAB"])
    for _, row in mutual_matches["Class"].iterrows()
}

predicted_object_properties = {
    (row["Wittgensteiniana"], row["WAB"])
    for _, row in mutual_matches["Object property"].iterrows()
}

class_metrics = calculate_metrics(
    predicted_classes,
    reference_classes
)

property_metrics = calculate_metrics(
    predicted_object_properties,
    reference_properties
)

metrics_df = pd.DataFrame([
    {
        "Category": "Classes",
        **{
            key: class_metrics[key]
            for key in ["TP", "FP", "FN", "Precision", "Recall", "F1-score"]
        }
    },
    {
        "Category": "Object properties",
        **{
            key: property_metrics[key]
            for key in ["TP", "FP", "FN", "Precision", "Recall", "F1-score"]
        }
    }
])

display(metrics_df.round(3))

,Category,TP,FP,FN,Precision,Recall,F1-score
0,Classes,0,3,6,0.00,0.0,0.000
1,Object properties,3,1,2,0.75,0.6,0.667


In [23]:
for category, result in [
    ("Classes", class_metrics),
    ("Object properties", property_metrics)
]:
    print(f"\n{'=' * 60}")
    print(category)

    print("\nCorrect matches:")
    for pair in result["Correct pairs"]:
        print(f"  {pair[0]} <-> {pair[1]}")

    print("\nFalse positives:")
    for pair in result["False positives"]:
        print(f"  {pair[0]} <-> {pair[1]}")

    print("\nMissed reference matches:")
    for pair in result["Missed pairs"]:
        print(f"  {pair[0]} <-> {pair[1]}")


Classes

Correct matches:

False positives:
  Author <-> Source
  Chapter <-> TS
  CreativeWork <-> Werk

Missed reference matches:
  Author <-> Person
  Book <-> Werk
  Chapter <-> Part
  CreativeWork <-> WittgensteinSource
  TextFragment <-> Bemerkung
  Thesis <-> Part

Object properties

Correct matches:
  hasAuthor <-> hasAuthor
  isPartOf <-> isPartOf
  mentions <-> refersTo

False positives:
  translationOfWork <-> language

Missed reference matches:
  inLanguage <-> language
  isPartOf <-> hasPart


In [24]:
def evaluate_top_k(df, reference, k_values=(1, 3, 5)):
    results = []

    for k in k_values:
        predicted = set()

        for source in df["WAB"].unique():
            candidates = (
                df[df["WAB"] == source]
                .sort_values("Similarity", ascending=False)
                .head(k)
            )

            for _, row in candidates.iterrows():
                predicted.add((
                    row["Wittgensteiniana"],
                    row["WAB"]
                ))

        metrics = calculate_metrics(predicted, reference)

        results.append({
            "K": k,
            "TP": metrics["TP"],
            "FP": metrics["FP"],
            "FN": metrics["FN"],
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "F1-score": metrics["F1-score"]
        })

    return pd.DataFrame(results)


class_top_k = evaluate_top_k(
    grouped_results["Class"],
    reference_classes
)

property_top_k = evaluate_top_k(
    grouped_results["Object property"],
    reference_properties
)

print("CLASS ALIGNMENT")
display(class_top_k.round(3))

print("OBJECT PROPERTY ALIGNMENT")
display(property_top_k.round(3))

CLASS ALIGNMENT


,K,TP,FP,FN,Precision,Recall,F1-score
0,1,1,7,5,0.125,0.167,0.143
1,3,4,20,2,0.167,0.667,0.267
2,5,5,35,1,0.125,0.833,0.217


OBJECT PROPERTY ALIGNMENT


,K,TP,FP,FN,Precision,Recall,F1-score
0,1,4,6,1,0.400,0.8,0.533
1,3,4,26,1,0.133,0.8,0.229
2,5,4,46,1,0.080,0.8,0.145
